In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 18
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.542255902061342

Trial 1 =========================================
14.097326400898611

Trial 2 =========================================
13.92969648636965

Trial 3 =========================================
14.869928056280072

Trial 4 =========================================
13.90426170750709



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 5 =========================================
15.929222010399386

Trial 6 =========================================
14.530241070586683

Trial 7 =========================================
14.62223887610584

Trial 8 =========================================
14.265504983519927

Trial 9 =========================================
16.07877143470119

Trial 10 =========================================
15.916840764772548

Trial 11 =========================================
15.271641109973498



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 12 =========================================
15.982179100564611

Trial 13 =========================================
14.51545620760283

Trial 14 =========================================
17.38518600287995

Trial 15 =========================================
15.308159415593119

Trial 16 =========================================
13.92456723603149

Trial 17 =========================================
14.29362574944554

Trial 18 =========================================
13.981804351327039

Trial 19 =========================================
14.263298155148885

Trial 20 =========================================
15.050749756937453

Trial 21 =========================================
17.561899326759036

Trial 22 =========================================
14.150436224807779

Trial 23 =========================================
13.891289005882967

Trial 24 =========================================
13.993123369348467

Trial 25 =========================================
16.129771417160416

Trial 26 =

...

Trial 85 =========================================
17.555590970694873

Trial 86 =========================================
18.704548013044842

Trial 87 =========================================
15.845986018086084

Trial 88 =========================================
13.397683053916735

Trial 89 =========================================
15.361494281198683

Trial 90 =========================================
14.200953551976543

Trial 91 =========================================
14.907952728085515

Trial 92 =========================================
16.142380916788383

Trial 93 =========================================
14.264648001958598

Trial 94 =========================================
14.062961895716303

Trial 95 =========================================
14.63590175965954

Trial 96 =========================================
16.59176723916508



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 97 =========================================
16.152404498376644



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 98 =========================================
16.1683692413113

Trial 99 =========================================
14.097229669768783

[13.5422559  14.0973264  13.92969649 14.86992806 13.90426171 15.92922201
 14.53024107 14.62223888 14.26550498 16.07877143 15.91684076 15.27164111
 15.9821791  14.51545621 17.385186   15.30815942 13.92456724 14.29362575
 13.98180435 14.26329816 15.05074976 17.56189933 14.15043622 13.89128901
 13.99312337 16.12977142 16.05595656 14.3427914  16.04236988 17.64714274
 16.06596912 15.8792485  16.23596878 15.35714589 18.65498381 14.61533313
 16.05004743 14.0126861  14.15297326 14.05423078 15.36744834 17.07722403
 15.46282373 15.3131174  13.86420459 18.34777599 16.19029658 14.34142613
 16.20191009 14.48066651 12.99971164 16.19874866 14.25683968 14.39979638
 16.10174099 13.42124352 14.25825574 16.22421383 13.62999075 14.59656094
 13.78813173 15.97985275 16.12439144 14.19091459 18.7522632  16.19026564
 14.44577162 13.99777387 13.90627112 13.60374248 16.00184

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.812610051763507
Avg = 15.245357308218061
Std = 1.3088539218727122


In [6]:
print(y_max_arr.tolist())

[13.542255902061342, 14.097326400898611, 13.92969648636965, 14.869928056280072, 13.90426170750709, 15.929222010399386, 14.530241070586683, 14.62223887610584, 14.265504983519927, 16.07877143470119, 15.916840764772548, 15.271641109973498, 15.982179100564611, 14.51545620760283, 17.38518600287995, 15.308159415593119, 13.92456723603149, 14.29362574944554, 13.981804351327039, 14.263298155148885, 15.050749756937453, 17.561899326759036, 14.150436224807779, 13.891289005882967, 13.993123369348467, 16.129771417160416, 16.055956561507926, 14.34279140298948, 16.042369883956624, 17.647142741351658, 16.06596911928944, 15.879248496122553, 16.235968776901363, 15.35714588931956, 18.654983809736205, 14.61533313440085, 16.050047433677907, 14.012686099062307, 14.152973260452915, 14.054230784511184, 15.367448344358289, 17.07722402506514, 15.462823732382827, 15.31311739791759, 13.86420458598426, 18.34777599105083, 16.19029657778528, 14.341426131719405, 16.201910090687136, 14.480666513936228, 12.9997116436210

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-18.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)